# Guía de Elementos y Conceptos — Práctica 1: Procesamiento Básico y Color

Esta guía resume los **elementos técnicos, funciones, tipos de datos y patrones de código** que se trabajan en la Práctica 1 (y su tutorial previo), organizados por bloques temáticos para consulta rápida durante el examen.

### Índice de Bloques
1. **[Representación Matricial, Propiedades y Niveles de Intensidad](#1-representacion)**
2. **[Tipos de Datos (`dtype`) y Conversiones (`uint8` vs `float`)](#2-tipos-y-conversiones)**
3. **[Creación de Imágenes Sintéticas, Lectura y Guardado (`io`)](#3-creacion-lectura-guardado)**
4. **[Visualización con Matplotlib y Control del Contraste (`vmin`, `vmax`)](#4-visualizacion)**
5. **[Indexación Espacial (Slicing), Volteos, Marcos y Redimensionado](#5-slicing-marcos-escala)**
6. **[Ajuste de Brillo y Estiramiento de Contraste (*Contrast Stretching*)](#6-brillo-y-contraste)**
7. **[Manipulación de Canales de Color (RGB y RGBA)](#7-canales-de-color)**
8. **[Umbralización (Binarización) y Máscaras Booleanas sin Bucles](#8-mascaras-booleanas)**

In [ ]:
# Imports estándar utilizados en toda la Práctica 1
%matplotlib inline
import os
import numpy as np
from skimage import io, img_as_float, img_as_ubyte
from skimage.color import rgb2gray, rgba2rgb
from skimage.exposure import rescale_intensity
from skimage.transform import resize, rescale
from matplotlib import pyplot as plt

<a id='1-representacion'></a>
---
## 1. Representación Matricial, Propiedades y Niveles de Intensidad

En NumPy y `scikit-image`, toda imagen es un array multidimensional donde el eje `0` son las **filas (alto / eje Y)**, el eje `1` son las **columnas (ancho / eje X)** y el eje `2` (si existe) son los **canales de color**.

| Tipo de Imagen | Dimensiones (`shape`) | Nº de dimensiones (`ndim`) | Descripción de los valores |
| :--- | :--- | :--- | :--- |
| **Binaria** | `(Alto, Ancho)` | `2` | `False` (`0`, negro) o `True` (`1`, blanco) |
| **Escala de Grises** | `(Alto, Ancho)` | `2` | Intensidad luminosa: `0` (negro) al máximo (`255` en `uint8` o `1.0` en `float`) |
| **RGB (Color)** | `(Alto, Ancho, 3)` | `3` | 3 canales apilados: `[0]=Rojo (R)`, `[1]=Verde (G)`, `[2]=Azul (B)` |
| **RGBA (Color + Alpha)** | `(Alto, Ancho, 4)` | `3` | 4 canales: `R, G, B` + `[3]=Alpha` (opacidad/transparencia) |

### Resolución Fotométrica (Niveles de Intensidad)
- Si cada píxel usa $b$ bits, puede representar **$2^b$ niveles distintos de intensidad**.
- En una imagen **`uint8`** ($b = 8$ bits sin signo): existen **$2^8 = 256$ niveles distintos**, en el rango entero **`[0, 255]`**.

In [ ]:
# Propiedades esenciales para inspeccionar cualquier imagen
def inspeccionar_imagen(img):
    print("Dimensiones (shape):", img.shape)       # (Alto, Ancho) o (Alto, Ancho, Canales)
    print("Nº de ejes (ndim):", img.ndim)          # 2 (gris/binaria) o 3 (RGB/RGBA)
    print("Tipo de dato (dtype):", img.dtype)      # uint8, float64, bool...
    print("Rango [min, max]:", img.min(), img.max())
    print("Intensidad media:", np.mean(img))
    print("Niveles únicos presentes:", len(np.unique(img)))

<a id='2-tipos-y-conversiones'></a>
---
## 2. Tipos de Datos (`dtype`) y Conversiones (`uint8` vs `float`)

Uno de los puntos más importantes de la práctica es controlar cómo cambian el tipo de dato (`dtype`) y el rango de valores al aplicar funciones de `scikit-image`:

1. **Diferencia entre `.astype(float)` e `img_as_float(img)`:**
   - `img.astype(float)` convierte el tipo a `float64`, pero **mantiene los valores originales** (si era `uint8` en `[0, 255]`, pasa a `float64` en `[0.0, 255.0]`). Para normalizarlo manualmente hay que dividir entre `255`: `img.astype(float) / 255`.
   - `img_as_float(img)` convierte el tipo a `float64` y **reescala automáticamente el rango entero `[0, 255]` al intervalo `[0.0, 1.0]`**.
2. **Funciones que convierten automáticamente la salida a `float64` en `[0.0, 1.0]`:**
   - `rgb2gray(img_rgb)`
   - `resize(img, (new_h, new_w))` y `rescale(img, factor)` (salvo que se indique `preserve_range=True`)
3. **Conversión de `float` `[0.0, 1.0]` a `uint8` `[0, 255]`:**
   - `img_as_ubyte(img_float)` multiplica por 255, redondea y convierte a `uint8`. Es **obligatorio antes de guardar imágenes con `io.imsave` en formatos como JPG**.
4. **Peligros de operar directamente en `uint8`:**
   - `uint8` no admite decimales (los trunca) ni valores fuera de `[0, 255]` (si restas por debajo de 0 o superas 255 sin cuidado, se produce *overflow* módulo 256). Si vas a multiplicar por decimales o hacer convoluciones, convierte antes a `float`.

In [ ]:
# Conversiones habituales entre tipos de datos
# De uint8 [0, 255] a float64 [0.0, 1.0]:
# img_float = img_as_float(img_uint8)        # o bien: img_uint8.astype(float) / 255

# De RGB uint8 (H, W, 3) a Escala de Grises float64 (H, W) en [0.0, 1.0]:
# img_gray = rgb2gray(img_rgb)

# De float64 [0.0, 1.0] a uint8 [0, 255] (para guardar en disco):
# img_uint8 = img_as_ubyte(img_gray)

<a id='3-creacion-lectura-guardado'></a>
---
## 3. Creación de Imágenes Sintéticas, Lectura y Guardado (`io`)

- **Crear imágenes desde cero:**
  - Negra en escala de grises: `np.zeros((H, W), dtype=np.uint8)`
  - Negra en color RGB: `np.zeros((H, W, 3), dtype=np.uint8)`
  - Blanca en `uint8`: `255 * np.ones((H, W), dtype=np.uint8)`
  - Del mismo tamaño y tipo que otra imagen `img`: `np.zeros_like(img)` o `255 * np.ones_like(img)`
- **Lectura y escritura en disco (`skimage.io`):**
  - Cargar: `img = io.imread('carpeta/archivo.png')`
  - Guardar: `io.imsave('output/archivo.png', img_uint8)`

In [ ]:
# Creación de directorio de salida si no existe y guardado seguro
if not os.path.exists("output"):
    os.mkdir("output")

# Ejemplo: crear imagen negra 100x100 uint8 y guardarla
img_negra = np.zeros((100, 100), dtype=np.uint8)
io.imsave(os.path.join("output", "negra.png"), img_negra)

<a id='4-visualizacion'></a>
---
## 4. Visualización con Matplotlib y Control del Contraste (`vmin`, `vmax`)

### El comportamiento de `plt.imshow()`
- En imágenes 2D (escala de grises o binarias), hay que indicar siempre `cmap='gray'` (de lo contrario Matplotlib aplica un mapa de color falso tipo *viridis*).
- **Auto-escalado por defecto:** Si **no** se especifican `vmin` y `vmax`, `plt.imshow(img, cmap='gray')` asigna el color negro al valor mínimo de `img` y el blanco al valor máximo de `img`. Esto **mejora artificialmente el contraste en pantalla** y oculta cambios de brillo.
- **Cómo mostrar la intensidad real sin auto-contraste:**
  - Para imágenes `uint8` (rango `[0, 255]`): `plt.imshow(img, cmap='gray', vmin=0, vmax=255)`
  - Para imágenes `float` normalizadas (rango `[0.0, 1.0]`): `plt.imshow(img, cmap='gray', vmin=0, vmax=1)`

In [ ]:
# Plantilla para mostrar varias imágenes en una fila (1xN subplots) sin ejes
def mostrar_comparativa(imagenes, titulos, vmin=None, vmax=None):
    n = len(imagenes)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    for ax, im, titulo in zip(axes, imagenes, titulos):
        ax.imshow(im, cmap='gray', vmin=vmin, vmax=vmax)
        ax.set_title(titulo)
        ax.axis('off') # Equivalente a ocultar xaxis y yaxis
    plt.tight_layout()
    plt.show()

<a id='5-slicing-marcos-escala'></a>
---
## 5. Indexación Espacial (Slicing), Volteos, Marcos y Redimensionado

### 5.1 Slicing por Regiones y Volteos (Efecto Espejo)
- **Regla básica:** Antes de modificar píxeles por slicing sobre una imagen que vas a reutilizar, haz una copia independiente con **`img_copia = img.copy()`**.
- **Acceso a un píxel:** `img[fila, col]` (en 2D) o `img[fila, col, canal]` (en 3D).
- **Acceso a sub-regiones:**
  - Mitad derecha: `img[:, W//2:]`
  - Mitad inferior: `img[H//2:, :]`
  - Cuadrante superior izquierdo: `img[:H//2, :W//2]`
- **Volteos (Flip):**
  - **Vertical (arriba ↔ abajo):** `np.flipud(img)` | `img[::-1, :]` | `np.flip(img, axis=0)`
  - **Horizontal (izquierda ↔ derecha):** `np.fliplr(img)` | `img[:, ::-1]` | `np.flip(img, axis=1)`

### 5.2 Enmarcar una Imagen (Centrar en un Lienzo Mayor y Pintar Bordes)
Para crear un marco alrededor de una imagen de tamaño `(height, width, 3)` aumentando su tamaño un porcentaje (ej. 15% → factor `1.15`):
1. Calcular nuevas dimensiones enteras: `new_height = int(height * 1.15)`, `new_width = int(width * 1.15)`.
2. Calcular la esquina superior izquierda donde empieza la imagen centrada usando división entera `//`:
   - `start_row = (new_height - height) // 2`
   - `start_col = (new_width - width) // 2`
3. Copiar la imagen en `[start_row : start_row + height, start_col : start_col + width]`.
4. Si se pide cambiar el color del marco, modificar las 4 franjas exteriores:
   - Arriba: `[:start_row, :]` | Abajo: `[start_row + height:, :]`
   - Izquierda: `[:, :start_col]` | Derecha: `[:, start_col + width:]`

### 5.3 Diferencia entre `resize` y `rescale` (`skimage.transform`)
- **`resize(img, (new_height, new_width))`:** Redimensiona la imagen a un alto y ancho exactos en píxeles. Si no se mantiene la proporción original alto/ancho, la imagen se deforma (estira o aplasta).
- **`rescale(img, factor)`:** Escala tanto alto como ancho multiplicándolos por `factor` (ej. `1.15` para +15%, `1/20` para reducir 20 veces), conservando siempre la relación de aspecto.

In [ ]:
# Patrón completo: Crear lienzo un 15% mayor, centrar imagen RGB y pintar marco verde
def crear_marco_color(img_rgb, factor=1.15, color_rgb=[0, 255, 0]):
    height, width, channels = img_rgb.shape
    new_height = int(height * factor)
    new_width = int(width * factor)

    # Lienzo negro RGB (3 canales, uint8)
    img_framed = np.zeros((new_height, new_width, 3), dtype=np.uint8)

    # Coordenadas de inicio para centrar
    start_row = (new_height - height) // 2
    start_col = (new_width - width) // 2

    # Incrustar imagen original en el centro
    img_framed[start_row:start_row + height, start_col:start_col + width] = img_rgb

    # Pintar las 4 bandas del marco exterior
    img_framed[:start_row, :] = color_rgb           # Borde superior
    img_framed[start_row + height:, :] = color_rgb  # Borde inferior
    img_framed[:, :start_col] = color_rgb           # Borde izquierdo
    img_framed[:, start_col + width:] = color_rgb   # Borde derecho
    return img_framed

<a id='6-brillo-y-contraste'></a>
---
## 6. Ajuste de Brillo y Estiramiento de Contraste (*Contrast Stretching*)

Cuando una imagen tiene bajo contraste (ej. sus píxeles solo ocupan el rango `[105, 159]` dentro de `[0, 255]`), podemos modificar sus intensidades de dos formas muy distintas:

### 6.1 Modificación de Brillo (Desplazamiento aditivo/sustractivo)
- **Oscurecer llevando el mínimo a `0`:** `dark_image = img - img.min()` (el rango pasa a `[0, max - min]`).
- **Aclarar llevando el máximo a `255`:** `bright_im = img + (255 - img.max())` (el rango pasa a `[255 - (max - min), 255]`).
- **Efecto:** Solo desplaza el histograma hacia la izquierda o la derecha; la diferencia `max - min` no cambia, por lo que **el contraste no mejora**.

### 6.2 Estiramiento de Contraste (*Contrast Stretching*)
Expande linealmente las intensidades para que el mínimo pase a ser el negro puro y el máximo pase a ser el blanco puro:

| Método | Expresión / Función | `dtype` y Rango Resultante | Cómo mostrar con `plt.imshow` |
| :--- | :--- | :--- | :--- |
| **1. Fórmula Matemática** | `img_norm = (img - img.min()) / (img.max() - img.min())` | `float64` en **`[0.0, 1.0]`** | `vmin=0, vmax=1` |
| **2. `skimage.exposure`** | `img_resc = rescale_intensity(img)` | Mantiene `uint8` en **`[0, 255]`** | `vmin=0, vmax=255` |

In [ ]:
# Resumen de ajustes de brillo y contraste sobre una imagen uint8 de bajo contraste (img_cell)
# 1. Oscurecer (mínimo = 0)
# dark_image = img_cell - img_cell.min()

# 2. Aclarar (máximo = 255)
# bright_im = img_cell + (255 - img_cell.max())

# 3. Contrast Stretching por fórmula -> float64 en [0.0, 1.0] (usar vmin=0, vmax=1)
# img_formula = (img_cell - img_cell.min()) / (img_cell.max() - img_cell.min())

# 4. Contrast Stretching con skimage -> uint8 en [0, 255] (usar vmin=0, vmax=255)
# img_rescale = rescale_intensity(img_cell)

<a id='7-canales-de-color'></a>
---
## 7. Manipulación de Canales de Color (RGB y RGBA)

- **Extraer canales individuales de una imagen RGB `(H, W, 3)`:**
  - `red = img[:, :, 0]`, `green = img[:, :, 1]`, `blue = img[:, :, 2]`
  - Al visualizar un canal individual en escala de grises (`cmap='gray'`), las zonas blancas corresponden a píxeles con alta contribución de ese color primario.
- **Combinar canales 2D en una imagen RGB 3D:**
  - Crear `img_rgb = np.zeros((H, W, 3), dtype=np.uint8)` y asignar `img_rgb[:, :, 0] = c1`, `[:, :, 1] = c2`, `[:, :, 2] = c3`.
  - **Colores secundarios por mezcla aditiva:**
    - Rojo + Verde = **Amarillo** `[255, 255, 0]`
    - Rojo + Azul = **Magenta** `[255, 0, 255]`
    - Verde + Azul = **Cian** `[0, 255, 255]`
    - Rojo + Verde + Azul = **Blanco** `[255, 255, 255]`
- **Intercambiar dos canales (ej. Rojo y Azul):**
  - Hacer copia `swapped = img.copy()` y cruzar `swapped[:, :, 0] = img[:, :, 2]` y `swapped[:, :, 2] = img[:, :, 0]`.
- **Trabajar con imágenes RGBA `(H, W, 4)`:**
  - Los 3 primeros canales son `img[:, :, :3]` (RGB) y el cuarto `img[:, :, 3]` es el canal **Alpha** (transparencia). Al cambiar colores o superponer objetos, se opera únicamente sobre `:3` para no alterar la opacidad.

<a id='8-mascaras-booleanas'></a>
---
## 8. Umbralización (Binarización) y Máscaras Booleanas sin Bucles

En NumPy **nunca se deben usar bucles `for` píxel a píxel** para filtrar o sustituir regiones; se utilizan condiciones lógicas vectorizadas (máscaras booleanas):

1. **Binarización de una imagen en escala de grises por umbral (ej. la media):**
   - `img_binary = img_gray > np.mean(img_gray)`
   - Genera un array 2D de tipo `bool` donde los píxeles que superan la media valen `True` (`1`, blanco) y el resto `False` (`0`, negro).
2. **Máscara booleana 2D de píxeles NO negros en una imagen RGB o RGBA:**
   - Un píxel es negro si sus tres canales R, G y B son `0`. Por tanto, un píxel **no es negro** si **alguno (`np.any`)** de sus 3 canales RGB (`:3`) es mayor que `0`:
   - `mask = np.any(img[:, :, :3] > 0, axis=2)` (o `!= 0`).
   - El parámetro `axis=2` colapsa el eje de los canales, devolviendo una máscara booleana 2D de tamaño `(Alto, Ancho)`.
3. **Aplicar la máscara para cambiar canales o superponer un objeto sobre un fondo:**
   - **Intercambiar canales R y B solo dentro de la máscara:**
     - `img_changed[mask, 0] = img[mask, 2]`
     - `img_changed[mask, 2] = img[mask, 0]`
   - **Superponer un objeto (`img_coat`) sobre otra imagen (`img_model`) respetando el canal Alpha:**
     - `img_out = img_model.copy()`
     - `img_out[mask, :3] = img_coat[mask, :3]`

In [ ]:
# Patrón general de enmascaramiento booleano y superposición en imágenes RGBA (sin bucles for)
def cambiar_color_y_superponer_rgba(img_fondo, img_objeto):
    # 1. Máscara 2D (H, W) donde el objeto NO es negro (evaluando solo RGB :3)
    mask = np.any(img_objeto[:, :, :3] > 0, axis=2)

    # 2. Intercambiar canal Rojo (0) y Azul (2) solo en los píxeles activos de la máscara
    objeto_modificado = img_objeto.copy()
    objeto_modificado[mask, 0] = img_objeto[mask, 2]
    objeto_modificado[mask, 2] = img_objeto[mask, 0]

    # 3. Superponer sobre el fondo modificando solo los canales RGB (:3) y dejando Alpha intacto
    resultado = img_fondo.copy()
    resultado[mask, :3] = objeto_modificado[mask, :3]
    return objeto_modificado, resultado